In [15]:
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.graph.message import add_messages
from langgraph.prebuilt.tool_node import ToolNode, tools_condition
from typing import  TypedDict, Annotated
from app.agent.tools.generate_docx import generate_word_doc_tool

In [16]:
from langchain_core.prompts import ChatPromptTemplate


# Define a template with roles and placeholders
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an who based on the query of user classify the query to the specific category and run that specific agent to execte that query."),
    ("human", "generate a word document on neural network")
])


# llm = ChatOllama(model = "granite4.1:3b")




In [17]:
from pydantic import BaseModel
from enum import Enum
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage

from app.model.model_manager import ModelManager

# Enum for query types
class QueryType(str, Enum):
    coding = "coding"
    summarization = "summarization"
    vision = "vision"

class QueryTypeChoice(BaseModel):
    choice: QueryType

# State definition
class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    task: str

# Instantiate model manager
provider = "ollama"  # or huggingface/lmstudio
manager = ModelManager(provider)

# Orchestrator node: classify query type
def orchestrator_node(state: State):
    llm = manager.get_model("multimodal")
    llm_with_query_type = llm.with_structured_output(QueryTypeChoice)
    res = llm_with_query_type.invoke(state["messages"])
    return {"messages": [res], "task": res.choice.value}

# Agent nodes
def coding_agent(state: State):
    llm = manager.get_model("coding")
    res = llm.invoke(state["messages"])
    return {"messages": [res]}

def summarization_agent(state: State):
    llm = manager.get_model("summarization")
    res = llm.invoke(state["messages"])
    return {"messages": [res]}

def vision_agent(state: State):
    llm = manager.get_model("vision")
    res = llm.invoke(state["messages"])
    return {"messages": [res]}

# Graph wiring
graph = StateGraph(State)
graph.add_node("orchestrator", orchestrator_node)
graph.add_node("coding_agent", coding_agent)
graph.add_node("summarization_agent", summarization_agent)
graph.add_node("vision_agent", vision_agent)

graph.add_edge(START, "orchestrator")

graph.add_conditional_edges(
    "orchestrator",
    lambda state: state["task"],
    {
        "coding": "coding_agent",
        "summarization": "summarization_agent",
        "vision": "vision_agent"
    }
)

graph.add_edge("coding_agent", END)
graph.add_edge("summarization_agent", END)
graph.add_edge("vision_agent", END)

app = graph.compile()
app

Provider name :  ollama
Discovered models:
0. granite4.1:3b
1. deepseek-coder:6.7b
2. qwen3-vl:4b
3. qwen2.5-coder:3b
4. embeddinggemma:300m
5. codesage:latest
6. llama3.2:3b


AttributeError: 'WindowsPath' object has no attribute 'read'